In [27]:
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from tqdm.auto import tqdm
tqdm.pandas()

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
df = pd.read_csv('/content/drive/MyDrive/NLP/Book Genre Prediction.csv')
df

,index,title,genre,summary
0,0,Drowned Wednesday,fantasy,Drowned Wednesday is the first Trustee among ...
1,1,The Lost Hero,fantasy,"As the book opens, Jason awakens on a school ..."
2,2,The Eyes of the Overworld,fantasy,Cugel is easily persuaded by the merchant Fia...
3,3,Magic's Promise,fantasy,The book opens with Herald-Mage Vanyel return...
4,4,Taran Wanderer,fantasy,Taran and Gurgi have returned to Caer Dallben...
...,...,...,...,...
4652,4652,Hounded,fantasy,"Atticus O’Sullivan, last of the Druids, lives ..."
4653,4653,Charlie and the Chocolate Factory,fantasy,Charlie Bucket's wonderful adventure begins wh...
4654,4654,Red Rising,fantasy,"""I live for the dream that my children will be..."
4655,4655,Frostbite,fantasy,"Rose loves Dimitri, Dimitri might love Tasha, ..."


In [30]:
df = df.drop(["index"], axis=1)

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4657 entries, 0 to 4656
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    4657 non-null   object
 1   genre    4657 non-null   object
 2   summary  4657 non-null   object
dtypes: object(3)
memory usage: 109.3+ KB


In [32]:
df['genre'].value_counts()

,count
genre,
thriller,1023
fantasy,876
science,647
history,600
horror,600
crime,500
romance,111
psychology,100
sports,100


In [33]:
df['genre'].value_counts()

,count
genre,
thriller,1023
fantasy,876
science,647
history,600
horror,600
crime,500
romance,111
psychology,100
sports,100


In [34]:
# Create a mapping from genre string to integer
unique_genres = df['genre'].unique()
genre_to_id = {genre: i for i, genre in enumerate(unique_genres)}
id_to_genre = {i: genre for i, genre in enumerate(unique_genres)}

# Create a numerical genre column
df['numerical_genre'] = df['genre'].map(genre_to_id)

# Display the mapping and head of df with new column
print("Genre to ID mapping:", genre_to_id)
display(df.head())

Genre to ID mapping: {'fantasy': 0, 'science': 1, 'crime': 2, 'history': 3, 'horror': 4, 'thriller': 5, 'psychology': 6, 'romance': 7, 'sports': 8, 'travel': 9}


,title,genre,summary,numerical_genre
0,Drowned Wednesday,fantasy,Drowned Wednesday is the first Trustee among ...,0
1,The Lost Hero,fantasy,"As the book opens, Jason awakens on a school ...",0
2,The Eyes of the Overworld,fantasy,Cugel is easily persuaded by the merchant Fia...,0
3,Magic's Promise,fantasy,The book opens with Herald-Mage Vanyel return...,0
4,Taran Wanderer,fantasy,Taran and Gurgi have returned to Caer Dallben...,0


In [35]:
df['numerical_genre'].value_counts()

,count
numerical_genre,
5,1023
0,876
1,647
3,600
4,600
2,500
7,111
6,100
8,100


In [36]:
X = df.drop(df[['numerical_genre', 'genre']], axis=1)
y = df['numerical_genre']

In [37]:
train_text, validation_text, train_labels, validation_labels = train_test_split(X, y, test_size=0.2, random_state=42)

In [38]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [39]:
train_encodings = tokenizer(list(train_text['summary']), truncation=True, padding=True)

In [40]:
validation_encodings = tokenizer(list(validation_text['summary']), truncation=True, padding=True)

In [41]:
from torch.utils.data import Dataset, DataLoader
import torch
from transformers import get_linear_schedule_with_warmup

In [42]:
class BookDataset(Dataset):
  def __init__(self, encoding, labels):
    self.encoding = encoding
    self.labels = labels
  def __len__(self):
    return len(self.labels)
  def __getitem__(self, index):
    item = {key: torch.tensor(val[index]) for key, val in self.encoding.items()}
    item['labels'] = torch.tensor(self.labels[index])
    return item

In [43]:
train_dataset = BookDataset(train_encodings, list(train_labels))
validation_dataset = BookDataset(validation_encodings, list(validation_labels))

In [44]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=16, shuffle=True)

In [45]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [46]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=10)
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [47]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [48]:
EPOCHS = 5

In [49]:
total_steps = len(train_loader) * EPOCHS

In [50]:
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [53]:
for epoch in range(EPOCHS+1):
  model.train()
  total_loss = 0
  for batch in tqdm(train_loader, desc="train"):
    optimizer.zero_grad()
    batch = {k:v.to(device) for k,v in batch.items()}
    outputs = model(**batch)
    loss = outputs[0] # Correct: outputs[0] is loss during training
    total_loss += loss.item()
    loss.backward()
    optimizer.step()
    scheduler.step()
  print(f'Epoch: {epoch}, Loss: {total_loss/len(train_loader)}')
  model.eval()
  predictions = []
  true_labels = []
  with torch.no_grad():
    for batch in tqdm(validation_loader, desc='Validation'): # Use validation_loader here
      batch = {k:v.to(device) for k,v in batch.items()}
      outputs = model(**batch)
      logits = outputs[1] # Corrected: logits are now outputs[1] during validation
      prediction = torch.argmax(logits, dim=1).cpu().numpy()
      predictions.extend(prediction)
      true_labels.extend(batch['labels'].cpu().numpy())
  Accuracy = accuracy_score(true_labels, predictions)
  print(f'Accuracy: {Accuracy:.4f}')

print(classification_report(true_labels, predictions))

train:   0%|          | 0/233 [00:00<?, ?it/s]

Epoch: 0, Loss: 0.29623949735589294


Validation:   0%|          | 0/59 [00:00<?, ?it/s]

Accuracy: 0.7629


train:   0%|          | 0/233 [00:00<?, ?it/s]

...

Accuracy: 0.7661


train:   0%|          | 0/233 [00:00<?, ?it/s]

Epoch: 5, Loss: 0.07957146401505102


Validation:   0%|          | 0/59 [00:00<?, ?it/s]

Accuracy: 0.7661
              precision    recall  f1-score   support

           0       0.81      0.84      0.82       196
           1       0.76      0.76      0.76       120
           2       0.77      0.72      0.74       109
           3       0.83      0.84      0.84       118
           4       0.66      0.62      0.64       113
           5       0.72      0.76      0.74       198
           6       0.86      0.69      0.77        26
           7       0.56      0.59      0.57        17
           8       0.93      1.00      0.96        13
           9       0.91      0.91      0.91        22

    accuracy                           0.77       932
   macro avg       0.78      0.77      0.78       932
weighted avg       0.77      0.77      0.77       932

